# S7 · Generative arm: stochastic-interpolant flow matching

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/hashirama21/sct-range/blob/main/notebooks/s7_generative.ipynb) [![Open In Kaggle](https://kaggle.com/static/images/open-in-kaggle.svg)](https://kaggle.com/kernels/welcome?src=https://github.com/hashirama21/sct-range/blob/main/notebooks/s7_generative.ipynb)

Weeks 7–8 of the plan build a generative sCT model with the backbone and budget of the regression arms and ask what it
adds. Every item of that block runs here:

| Plan item | Section |
|---|---|
| FM source → CT with a stochastic interpolant, same backbone; sweep η ∈ {0, 0.05, 0.1, 0.2} | 2, 3 |
| quality against the number of steps (1, 2, 4, 8, 16, 32) | 4 |
| projected CRPS, coverage, uncertainty–error correlation | 5 |
| generative vs the R6 head on tails; generative vs R3 on the mean (the theory predicts equality at best) | 6 |

**Model** (`sctrange.generative`, `configs/s7_preregistration.json`). In encoded CT units,
x_t = (1 − t) x₀ + t x₁ + γ(t) z with γ(t) = 2η t(1 − t), x₀ the bulk-density image (water in the body), x₁ the
reference CT and z Gaussian noise inside the body. The U-Net of R1–R6, with FiLM time conditioning, predicts the drift
and, for η > 0, the noise; samples come from Euler–Maruyama from x₀. With η = 0 sampling is deterministic and the
ensemble collapses to one image, and a single step returns the conditional mean in HU: the sweep shows what
the stochastic part buys.

**Point prediction.** The SPR mean of the samples, g⁻¹(mean g(sample)), is the Bayes act of the squared range loss
(Proposition 4); it is evaluated with the S4 metrics. Ensembles are scored on the WET error of every ray from entry to
the distal target edge: projected CRPS (proper, not strictly proper on finite segment families, Proposition 8),
coverage of central 90 % intervals, PIT, and the correlation between ensemble spread and the error of the mean.

**Compute.** 4 generators × 5 folds × 30 000 steps (about as much as one S4 seed), then sampling: K samples × N steps
per slice. Sampling is the costly part, so the pre-registration fixes nested case subsets (30 for the sweep, 10 for the
step curve, 100 for the main comparison). Every fold checkpoint and every evaluated case is stored and reused.

The notebook runs on the **real SynthRAD2025 training data** (Zenodo record 15373853), read from its
Hugging Face mirror [developper669/synthrad2025](https://huggingface.co/datasets/developper669/synthrad2025)
(tag `v2`). `SYNTHRAD2025_ROOT` points to the local copy (default `data/synthrad2025` in the repository,
ignored by git); missing tasks are fetched first (`sctrange.download`, parallel and resumable, only the
requested task; `SCT_DOWNLOAD=0` forbids it and stops with an error instead).

### Setup · install `sctrange` from the repository

Run this cell first. Inside a clone of [hashirama21/sct-range](https://github.com/hashirama21/sct-range) it installs that
clone in editable mode (`pip install -e ".[dose]"`), so the notebook always runs the repository's code, local edits
included. Opened on its own, from the Colab or Kaggle badge or on another machine, it first downloads the repository
from GitHub (no git needed) to `/content/sct-range` on Colab, `/kaggle/tmp/sct-range` on Kaggle (so that
`/kaggle/working` only receives outputs) or `./sct-range` elsewhere, then installs it. On Kaggle, enable
*Settings → Internet* first. `SCTRANGE_REF` selects a branch, tag or commit (default `main`), `SCTRANGE_HOME` the
destination; pip only runs when `sctrange` is missing or installed from elsewhere (`SCTRANGE_REINSTALL=1` forces it).

In [ ]:
import importlib
import importlib.util
import io
import os
import shutil
import subprocess
import sys
import time
import urllib.request
import zipfile
from pathlib import Path

REPO_URL = "https://github.com/hashirama21/sct-range"
REPO_REF = os.environ.get("SCTRANGE_REF", "main")
ON_COLAB = "google.colab" in sys.modules
ON_KAGGLE = "KAGGLE_KERNEL_RUN_TYPE" in os.environ or Path("/kaggle/working").is_dir()


def find_repo(start: Path) -> Path | None:
    return next((p for p in (start, *start.parents) if (p / "pyproject.toml").exists() and (p / "src" / "sctrange").is_dir()),
                None)


def download_repo(destination: Path) -> Path:
    """Fetch ``REPO_REF`` of the GitHub repository as a zip archive and unpack it to ``destination``."""
    for attempt in range(4):
        try:
            with urllib.request.urlopen(f"{REPO_URL}/archive/{REPO_REF}.zip", timeout=120) as response:
                archive = zipfile.ZipFile(io.BytesIO(response.read()))
            break
        except OSError as error:
            if attempt == 3:
                raise
            print(f"download failed ({error}); retrying")
            time.sleep(5 * (attempt + 1))
    top = archive.namelist()[0].split("/")[0]
    staging = destination.with_name(destination.name + ".partial")
    shutil.rmtree(staging, ignore_errors=True)
    archive.extractall(staging)
    destination.parent.mkdir(parents=True, exist_ok=True)
    if destination.exists():  # an earlier, interrupted attempt or another folder
        if any(destination.iterdir()):
            raise RuntimeError(f"{destination} exists and is not a clone of {REPO_URL}; remove it or set SCTRANGE_HOME")
        destination.rmdir()
    (staging / top).rename(destination)
    shutil.rmtree(staging, ignore_errors=True)
    return destination


REPO_ROOT = find_repo(Path.cwd())
if REPO_ROOT is None:
    default_home = (Path("/content/sct-range") if ON_COLAB else Path("/kaggle/tmp/sct-range") if ON_KAGGLE
                    else Path.cwd() / "sct-range")
    REPO_ROOT = Path(os.environ.get("SCTRANGE_HOME", default_home)).resolve()
    if find_repo(REPO_ROOT) is None:
        print(f"downloading {REPO_URL} @ {REPO_REF} to {REPO_ROOT}")
        download_repo(REPO_ROOT)
    os.chdir(REPO_ROOT / "notebooks")
SOURCE_DIR = (REPO_ROOT / "src").resolve()


def installed_from_repo() -> bool:
    spec = importlib.util.find_spec("sctrange")
    return spec is not None and spec.origin is not None and Path(spec.origin).resolve().is_relative_to(SOURCE_DIR)


if os.environ.get("SCTRANGE_REINSTALL") == "1" or not installed_from_repo():
    command = [sys.executable, "-m", "pip", "install", "--quiet", "--editable", f"{REPO_ROOT}[dose]"]
    for attempt in range(1, 4):
        result = subprocess.run(command, capture_output=True, text=True)
        if result.returncode == 0:
            break
        print(f"pip install failed (attempt {attempt}/3):\n{result.stderr[-3000:]}")
        time.sleep(10 * attempt)
    else:
        raise RuntimeError("pip could not install sctrange; see the messages above (network? Kaggle: enable Internet)")
# an editable install is only picked up by new processes; make it visible to this kernel too
if str(SOURCE_DIR) not in sys.path:
    sys.path.insert(0, str(SOURCE_DIR))
for module in [name for name in sys.modules if name == "sctrange" or name.startswith("sctrange.")]:
    del sys.modules[module]
importlib.invalidate_caches()

import sctrange

assert Path(sctrange.__file__).resolve().is_relative_to(SOURCE_DIR), f"sctrange imported from {sctrange.__file__}"
print(f"sctrange {sctrange.__version__} installed from {REPO_ROOT.name}/src (editable)")

### Parameters

Set them here and run all cells (*Runtime → Run all* on Colab, where this cell is a form). Environment variables of the
same meaning, given in brackets, override them for unattended runs. The verdict assumes the pre-registered defaults;
smaller values give a quick pass only. Training needs a GPU (T4 or 2 × T4); with two GPUs folds train in parallel.

| Platform | Dataset, cache | Outputs |
|---|---|---|
| local | `data/synthrad2025`, `outputs/` | `outputs/` |
| Colab | `/content/data`, `/content/cache` | Google Drive `MyDrive/sct-range/outputs` (`USE_DRIVE`) |
| Kaggle | `/kaggle/tmp/data`, `/kaggle/tmp/cache` | `/kaggle/working/outputs` |

In [ ]:
# @title Parameters { display-mode: "form" }
TASK = "1"  # @param ["1", "2"]  [SCT_TASK]
DOWNLOAD = True  # @param {type:"boolean"}  fetch missing data from Hugging Face [SCT_DOWNLOAD]
TRAIN = True  # @param {type:"boolean"}  train missing folds here (needs a GPU) [SCT_TRAIN_IN_NOTEBOOK]
STEPS = 30000  # @param {type:"integer"}  optimisation steps per fold [SCT_STEPS]
FOLDS = 5  # @param {type:"integer"}  [SCT_FOLDS]
ETAS = "0,0.05,0.1,0.2"  # @param {type:"string"}  [SCT_ETAS]
SWEEP_CASES = 30  # @param {type:"integer"}  [SCT_SWEEP_CASES]
STEP_CURVE_CASES = 10  # @param {type:"integer"}  [SCT_STEP_CASES]
MAIN_CASES = 100  # @param {type:"integer"}  [SCT_MAIN_CASES]
USE_DRIVE = True  # @param {type:"boolean"}  Colab only

In [ ]:
import hashlib
import json
import os
import warnings
from dataclasses import asdict, replace

warnings.filterwarnings("ignore", message=r"`torch\.jit\.script` is deprecated")

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
from IPython.display import display

from sctrange.cache import VolumeCache
from sctrange.data import discover_cases, load_case, read_prediction, read_volume
from sctrange.download import ensure_dataset, usable_records
from sctrange.runs import run_complete
from sctrange.evaluation import DiagnosticConfig, evaluate_case, evaluate_risk
from sctrange.generative import (GenerativeConfig, _load_generator, comparison_frame, crossfit_generative,
                                 collapse_ratio, decide_generative, evaluate_generative, run_name, sample_case,
                                 select_eta, spr_mean)
from sctrange.hlut import HLUT
from sctrange.plotting import (apply_style, plot_eta_sweep, plot_paired_differences, plot_pit, plot_sample_gallery,
                               plot_steps_curve, plot_training_curves)
from sctrange.runtime import frozen_split_path, resolve_paths
from sctrange.split import load_selection, load_split, select_cases, split_hash, write_selection
from sctrange.training import visible_devices

In [ ]:
PATHS = resolve_paths(REPO_ROOT, USE_DRIVE)
TASK = os.environ.get("SCT_TASK", TASK)
DOWNLOAD = os.environ.get("SCT_DOWNLOAD", "1" if DOWNLOAD else "0") != "0"
TRAIN = os.environ.get("SCT_TRAIN_IN_NOTEBOOK", "1" if TRAIN else "0") == "1"
STEPS = int(os.environ.get("SCT_STEPS", STEPS))
N_FOLDS = int(os.environ.get("SCT_FOLDS", FOLDS))
ETAS = tuple(float(e) for e in os.environ.get("SCT_ETAS", ETAS).split(","))
SWEEP_CASES = int(os.environ.get("SCT_SWEEP_CASES", SWEEP_CASES))
STEP_CURVE_CASES = int(os.environ.get("SCT_STEP_CASES", STEP_CURVE_CASES))
MAIN_CASES = int(os.environ.get("SCT_MAIN_CASES", MAIN_CASES))

OUTPUT_DIR = PATHS.output_root / "s7"
FIGURE_DIR = OUTPUT_DIR / "figures"
RUNS_ROOT = Path(os.environ.get("SCT_GEN_RUNS_ROOT", OUTPUT_DIR / "runs"))
S4_RUNS = Path(os.environ.get("SCT_S4_RUNS_ROOT", PATHS.output_root / "s4" / "runs"))
CACHE_DIR = Path(os.environ.get("SCT_CACHE_DIR", PATHS.cache_root / "cache"))
FROZEN_SPLIT_PATH = frozen_split_path(REPO_ROOT, PATHS)
PREREGISTRATION_PATH = REPO_ROOT / "configs" / "s7_preregistration.json"

prereg_bytes = PREREGISTRATION_PATH.read_bytes()
PREREG = json.loads(prereg_bytes)
PREREG_SHA256 = hashlib.sha256(prereg_bytes).hexdigest()
SEED = 20261013
DEVICES = visible_devices()
DEVICE = DEVICES[0]
DIAGNOSTIC = DiagnosticConfig(angles_deg=tuple(range(0, 360, 45)), target_radius_mm=25.0, seed=SEED, device=DEVICE)
BASE_CONFIG = GenerativeConfig(steps=STEPS, seed=PREREG["seed"], device=DEVICE)
hlut = HLUT.pyradplan_default()

DATA_ROOT = ensure_dataset(PATHS.data_root, (TASK,), download=DOWNLOAD)
TRAIN_IN_NOTEBOOK = TRAIN and (DEVICE.startswith("cuda") or os.environ.get("SCT_TRAIN_ON_CPU") == "1")
if TRAIN and not TRAIN_IN_NOTEBOOK:
    print("no GPU: training skipped, existing checkpoints only (SCT_TRAIN_ON_CPU=1 forces CPU training)")
torch.manual_seed(SEED)
FIGURE_DIR.mkdir(parents=True, exist_ok=True)
apply_style()
pd.set_option("display.width", 160)
display(pd.Series(PATHS.describe(REPO_ROOT), name="location").to_frame())
defaults = (PREREG["steps"], PREREG["folds"], tuple(PREREG["eta_grid"]), PREREG["sweep"]["cases"],
            PREREG["steps_curve"]["cases"], PREREG["main"]["cases"])
if (STEPS, N_FOLDS, ETAS, SWEEP_CASES, STEP_CURVE_CASES, MAIN_CASES) != defaults:
    print("non-default budget: quick pass only, the verdict is not the pre-registered one")
print(f"task {TASK} · devices {', '.join(DEVICES)} · eta {ETAS} · steps {STEPS} · folds {N_FOLDS}")

## 1 · Frozen split, pre-registration and case subsets

The three evaluation subsets are drawn once with the pre-registered seed (region-stratified, hashed order within
region) from the development cases, written with their SHA-256 under `outputs/s7/subsets/` and reused on every run: a
rerun that would select other cases stops instead of silently reselecting. Every case has seed-0 R3 and R6 predictions
once S4 is finished, so every comparison is paired. Because
the order within a region is fixed, the smaller subsets are, up to the allocation by region, the first cases of the
larger ones.

In [ ]:
if not FROZEN_SPLIT_PATH.exists():
    raise FileNotFoundError(f"{FROZEN_SPLIT_PATH} is missing: run the S1 notebook once on the real data to freeze it")
split = load_split(FROZEN_SPLIT_PATH)
development_ids = set(split["development"])
assert not development_ids & set(split["test"])
development, excluded = usable_records(DATA_ROOT, [r for r in discover_cases(DATA_ROOT)
                                                  if r.case_id in development_ids and r.task == TASK])

prediction = lambda arm, r: S4_RUNS / "seed0" / arm / "predictions" / f"{r.case_id}.mha"
HAVE_S4 = all(run_complete(S4_RUNS / "seed0" / arm, [r.case_id for r in development]) for arm in ("R3", "R6"))
if not HAVE_S4:
    print(f"seed-0 R3/R6 runs not finished under {S4_RUNS}: the generator is evaluated alone, section 6 is skipped")
seed = PREREG["subsets"]["seed"]
subset_ids = {}
for name, n in (("sweep", SWEEP_CASES), ("steps", STEP_CURVE_CASES), ("main", MAIN_CASES)):
    path = OUTPUT_DIR / "subsets" / f"{name}.json"
    meta = {"task": TASK, "seed": seed, "n_cases": n, "split_sha256": split_hash(split)}
    write_selection(select_cases(development, n, seed), path, meta)  # refuses to change a frozen subset
    subset_ids[name] = load_selection(path)["case_ids"]
subsets = {name: [r for r in development if r.case_id in set(ids)] for name, ids in subset_ids.items()}
display(pd.Series({"split sha256": split_hash(split), "pre-registration sha256": PREREG_SHA256,
                   "development cases": len(development), **{f"{k} cases": len(v) for k, v in subsets.items()}},
                  name="value").to_frame())

## 2 · Cross-fitted training, one generator per η

Each generator is cross-fitted on the whole development pool of the task (5 stratified folds, seed 0), so that every
evaluated case is sampled by a model that never saw it. Checkpoints are reused when their configuration and training cases
match and refused otherwise; a fold cut mid-training resumes from its last in-fold save. Sampled cases are cached under
`runs/<eta>/eval/`, so the sections below also resume after an interruption. The losses are the drift and denoiser regressions, averaged over body pixels.

In [ ]:
cache = VolumeCache(CACHE_DIR)
print(f"{cache.build(development)} new cases cached")
configs = {eta: replace(BASE_CONFIG, eta=eta) for eta in ETAS}
histories, missing = {}, []
for eta, config in configs.items():
    run_dir = RUNS_ROOT / run_name(config)
    if TRAIN_IN_NOTEBOOK:
        crossfit_generative(development, cache, config, N_FOLDS, run_dir, DEVICES)
    if not all((run_dir / f"fold{k}.pt").exists() for k in range(N_FOLDS)):
        missing.append(run_name(config))
    elif (run_dir / "history.csv").exists():
        histories[f"η = {eta:g}"] = pd.read_csv(run_dir / "history.csv")
if missing:
    raise RuntimeError(f"missing generators: {', '.join(missing)}; run this notebook on a GPU with TRAIN = True, "
                       "or `sctrange-generative --eta ...`")
display(pd.Series(asdict(BASE_CONFIG), name="configuration").to_frame())
plot_training_curves(histories).savefig(FIGURE_DIR / "training_curves.png", bbox_inches="tight")
plt.show()

## 3 · η sweep and pre-registered choice of η

Every generator samples the sweep cases (K = 8, N = 16). Among the stochastic generators (η > 0), the selected η
minimises the median projected CRPS (ties go to the smaller η) and is used in every later section. η = 0 is the
control: it is deterministic, so its ensemble spread must be at most 5 % of the smallest spread among η > 0 (collapse
check), and its CRPS equals the absolute WET error of its single image.

In [ ]:
sweep_spec = PREREG["sweep"]
sweep_frames, sweep_pits = [], {}
for eta, config in configs.items():
    rows, pits = evaluate_generative(subsets["sweep"], cache, RUNS_ROOT / run_name(config), config, hlut, DIAGNOSTIC,
                                     sweep_spec["n_samples"], sweep_spec["n_steps"], DEVICE, full=False)
    sweep_frames.append(rows)
    sweep_pits[eta] = pits
sweep = pd.concat(sweep_frames, ignore_index=True)
sweep.to_csv(OUTPUT_DIR / "s7_sweep.csv", index=False)
SELECTED_ETA = select_eta(sweep, sweep_spec["selection_metric"])
summary = sweep.groupby("eta")[["mae_hu", "target_rms_mm", "pcrps_mm", "pcrps_point_mm", "coverage90", "spread_mm",
                                "unc_err_spearman"]].median()
display(summary.round(3))
COLLAPSE_RATIO = collapse_ratio(sweep)
COLLAPSES = bool(np.isfinite(COLLAPSE_RATIO) and COLLAPSE_RATIO <= PREREG["collapse"]["max_spread_ratio"])
print(f"selected eta = {SELECTED_ETA:g} · spread ratio eta 0 / eta > 0 = {COLLAPSE_RATIO:.3g} "
      f"({'collapses' if COLLAPSES else 'does not collapse'})")
plot_eta_sweep(sweep, SELECTED_ETA).savefig(FIGURE_DIR / "eta_sweep.png", bbox_inches="tight")
plt.show()

## 4 · Quality against the number of steps

The selected generator samples the step-curve cases with 1, 2, 4, 8, 16 and 32 steps (K = 8). With one step every
sample is the conditional mean in HU, so the curve runs from a regression-like estimate (left) to samples of the
learned conditional law (right).

In [ ]:
curve_spec = PREREG["steps_curve"]
selected = configs[SELECTED_ETA]
curve = pd.concat([evaluate_generative(subsets["steps"], cache, RUNS_ROOT / run_name(selected), selected, hlut,
                                       DIAGNOSTIC, curve_spec["n_samples"], n, DEVICE, full=False)[0]
                   for n in curve_spec["n_steps"]], ignore_index=True)
curve.to_csv(OUTPUT_DIR / "s7_steps_curve.csv", index=False)
display(curve.groupby("n_steps")[["mae_hu", "sample_mae_hu", "target_rms_mm", "pcrps_mm", "coverage90",
                                  "spread_mm"]].median().round(3))
plot_steps_curve(curve).savefig(FIGURE_DIR / "steps_curve.png", bbox_inches="tight")
plt.show()

## 5 · Main evaluation: calibration of the range-error distribution

The selected generator samples the main cases (K = 16, N = 16). Its SPR-mean prediction is written to
`runs/<eta>/predictions/` (it can be dosed like an S4 arm) and evaluated with the S4 metrics, permutation controls
included. A calibrated ensemble gives a flat PIT histogram, coverage near 0.9 and a positive spread–error correlation.

In [ ]:
main_spec = PREREG["main"]
main_rows, main_pits = evaluate_generative(subsets["main"], cache, RUNS_ROOT / run_name(selected), selected, hlut,
                                           DIAGNOSTIC, main_spec["n_samples"], main_spec["n_steps"], DEVICE,
                                           write_mean=True, full=True)
main_rows.to_csv(OUTPUT_DIR / "s7_main.csv", index=False)
calibration = main_rows.groupby("region")[["coverage90", "spread_mm", "pcrps_mm", "pcrps_point_mm",
                                           "unc_err_spearman", "risk_coverage"]].median()
calibration.loc["all"] = main_rows[calibration.columns].median()
display(calibration.round(3))
print(f"cases with positive spread–error correlation: {(main_rows['unc_err_spearman'] > 0).mean():.0%}")
plot_pit(main_pits).savefig(FIGURE_DIR / "pit.png", bbox_inches="tight")
plt.show()

example = subsets["main"][0]
folds = pd.read_csv(RUNS_ROOT / run_name(selected) / "folds.csv", dtype={"case_id": str}).set_index("case_id")["fold"]
model = _load_generator(RUNS_ROOT / run_name(selected) / f"fold{folds[example.case_id]}.pt", replace(selected, device=DEVICE))
case = cache.load(example.case_id)
gallery = sample_case(model, case, selected, 4, main_spec["n_steps"], DEVICE, seed=1)
centre = int(np.argmax(np.asarray(case.mask).sum(axis=(1, 2))))
plot_sample_gallery(np.asarray(case.ct, dtype=np.float32), gallery, spr_mean(gallery, hlut), np.asarray(case.mask),
                    centre).savefig(FIGURE_DIR / "samples.png", bbox_inches="tight")
plt.show()

## 6 · Generative arm against R3 (mean) and R6 (tails)

R3 and R6 (seed 0, out of fold) are evaluated on the same main cases with the S4 metrics and, for R6, its risk maps.
A point prediction is a degenerate ensemble, so its projected CRPS is its mean absolute WET error. Pre-registered
tests (Holm over the four):

| Test | Column | Alternative | Theory |
|---|---|---|---|
| mean better than R3 | RMS ΔWET of the point prediction | G < R3 | not expected: equality at best |
| proper score better than R3 | projected CRPS | G < R3 | expected if the spread is informative |
| tail score vs R6 | FZ0 score of (VaR90, ES90) of the point prediction's \|ΔWET\| | two-sided | open |
| tail calibration vs R6 | \|coverage of VaR90 − 0.9\| | two-sided | open |

Each model forecasts the tail of its own error, so the FZ0 scores of G and R6 concern different outcomes; the
calibration test compares them on a common scale.

In [ ]:
if HAVE_S4:
    reference_path = OUTPUT_DIR / "s7_reference_arms.csv"
    risk_path = OUTPUT_DIR / "s7_reference_risk.csv"
    done = pd.read_csv(reference_path) if reference_path.exists() else pd.DataFrame(columns=["case_id", "source"])
    risk_done = pd.read_csv(risk_path) if risk_path.exists() else pd.DataFrame(columns=["case_id", "source"])
    ref_rows, risk_rows = [done], [risk_done]
    for record in subsets["main"]:
        if record.case_id in set(done["case_id"]):
            continue
        volumes = load_case(record, ("ct", "mask"))
        ct, mask, spacing = volumes["ct"].array, volumes["mask"].array, volumes["ct"].spacing
        keys = {"case_id": record.case_id, "region": record.region, "center": record.center}
        rows = []
        for arm in ("R3", "R6"):
            sct = read_prediction(prediction(arm, record), ct.shape)
            rows.append({**keys, "source": arm, **evaluate_case(ct, mask, sct, spacing, hlut, DIAGNOSTIC)})
        run = S4_RUNS / "seed0" / "R6" / "risk"
        if (run / f"{record.case_id}_var.mha").exists():
            var = read_volume(run / f"{record.case_id}_var.mha").array
            excess = read_volume(run / f"{record.case_id}_excess.mha").array
            sct = read_prediction(prediction("R6", record), ct.shape)
            risk_rows.append(pd.DataFrame([{**keys, "source": "R6",
                                            **evaluate_risk(ct, mask, sct, var, excess, spacing, hlut, DIAGNOSTIC)}]))
        ref_rows.append(pd.DataFrame(rows))
    reference = pd.concat(ref_rows, ignore_index=True)
    reference.to_csv(reference_path, index=False)
    reference_risk = pd.concat(risk_rows, ignore_index=True)
    reference_risk.to_csv(risk_path, index=False)

    frame = comparison_frame(main_rows, reference, reference_risk, PREREG["generative_name"])
    frame.to_csv(OUTPUT_DIR / "s7_comparison.csv", index=False)
    decision = decide_generative(frame, PREREG, sweep)
    display(frame.groupby("source")[["mae_hu", "target_rms_mm", "target_mean_signed_mm", "target_cvar_mm", "pcrps_mm",
                                     "risk_fz", "risk_coverage", "risk_spearman"]].median().round(3))
    display(pd.DataFrame(decision["tests"]).T[["column", "first", "second", "alternative", "cases",
                                               "median_difference", "p_value", "p_holm", "significant"]])
    display(pd.Series({k: v for k, v in decision.items() if k != "tests"}, name="value").to_frame())
    plot_paired_differences(frame, [("target_rms_mm", "G", "R3", "RMS ΔWET (mm)"),
                                    ("pcrps_mm", "G", "R3", "projected CRPS (mm)"),
                                    ("risk_fz", "G", "R6", "FZ0 tail score"),
                                    ("risk_miscalibration", "G", "R6", "|VaR90 coverage − 0.9|")]).savefig(
        FIGURE_DIR / "paired_differences.png", bbox_inches="tight")
    plt.show()
else:
    decision = {"tests": {}, "note": "no S4 runs: comparison not evaluated"}

## 7 · Report

In [ ]:
report = {
    "task": TASK,
    "split_sha256": split_hash(split),
    "preregistration_sha256": PREREG_SHA256,
    "subsets": subset_ids,
    "config": asdict(BASE_CONFIG),
    "etas": list(ETAS),
    "selected_eta": SELECTED_ETA,
    "collapse_ratio": COLLAPSE_RATIO,
    "sweep_medians": summary.to_dict(orient="index"),
    "steps_curve_medians": curve.groupby("n_steps")[["mae_hu", "target_rms_mm", "pcrps_mm"]].median()
                                .to_dict(orient="index"),
    "calibration": calibration.to_dict(orient="index"),
    "decision": decision,
}
(OUTPUT_DIR / "s7_report.json").write_text(json.dumps(report, indent=2, default=float))
status = pd.DataFrame([
    ("Frozen split verified", split_hash(split)[:12], True),
    ("Generators trained", ", ".join(run_name(c) for c in configs.values()), not missing),
    ("η selected (pre-registered rule)", f"{SELECTED_ETA:g}", True),
    ("η = 0 collapses", f"spread ratio {COLLAPSE_RATIO:.3g}", COLLAPSES),
    ("Steps curve", f"{curve['n_steps'].nunique()} step counts", True),
    ("Main evaluation", f"{len(main_rows)} cases", len(main_rows) > 0),
    ("Comparison with R3 and R6", "done" if HAVE_S4 else "no S4 runs", HAVE_S4),
], columns=["item", "detail", "passed"])
display(status)

## Reading the results

| Outcome | Meaning for the paper |
|---|---|
| mean not better than R3, proper score better | The theory's prediction: the generator adds a calibrated distribution, not a better point estimate. Result C (tails need the joint law) gets its empirical arm. |
| mean better than R3 | The generator is a better-fitted model, not a better estimator; check R3's capacity and training before claiming anything. |
| proper score not better than R3 | The spread is uninformative or miscalibrated (PIT shape tells which); the generative arm stays a negative result. |
| G better calibrated than R6 on tails | Tail estimation from samples beats the plan-independent density head; R6 remains the cheap option. |

The plan's S8 week continues this arm without new code: the main evaluation for task 2 (CBCT, `TASK = "2"`), the
selected generator's SPR mean dosed with the S6 notebook, and a larger K on the cases where coverage is off.